In [ ]:
# 03 - FLAN-T5 Arabic Tashkeel Inference

# ⚠️ WARNING: This notebook is BROKEN
# Model: Abdou/arabic-tashkeel-flan-t5-small
# Despite being "fine-tuned", produces 98.94% WER (essentially broken).
# The model generates garbage output regardless of prompt format.
# 
# DO NOT RUN - Results will be unusable.

raise RuntimeError("""
╔══════════════════════════════════════════════════════════════╗
║  NOTEBOOK DISABLED: FLAN-T5 PRODUCES 98.94% WER             ║
╠══════════════════════════════════════════════════════════════╣
║  Model: Abdou/arabic-tashkeel-flan-t5-small                  ║
║  Despite claiming to be fine-tuned for tashkeel, this model ║
║  produces nearly 99% word error rate (essentially broken).   ║
║                                                              ║
║  Use one of these working alternatives instead:              ║
║  - 01_infer_fine_tashkeel.ipynb (Fine-Tashkeel) ~42% WER    ║
║  - 02_infer_mushkil_arat5.ipynb (Mushkil-AraT5)             ║
║  - 04_infer_catt_tashkeel.ipynb (CATT) ~41% WER             ║
╚══════════════════════════════════════════════════════════════╝
""")

In [1]:
!pip install -q transformers torch accelerate sentencepiece tqdm

In [2]:
# Setup - Import from config.py (generated by setup.sh)
import os, sys, json, re, torch, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for config import
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Import paths from config
try:
    from config import (
        PROJECT_DIR, DATA_DIR, DEVICE,
        DEV_INPUT, DEV_OUTPUT,
        TEST_DIR, OUTPUT_DIR, SUBMISSION_DIR
    )
    TEST_INPUT = TEST_DIR / 'test_input.json'
except ImportError:
    print("WARNING: config.py not found. Run setup.sh first!")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    TEST_DIR = PROJECT_DIR / 'Test'
    TEST_INPUT = TEST_DIR / 'test_input.json'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


In [3]:
MODEL_KEY = 'flan_t5'
MODEL_NAME = 'Abdou/arabic-tashkeel-flan-t5-small'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
if device != "cuda":
    model = model.to(device)
model.eval()
print(f"Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading Abdou/arabic-tashkeel-flan-t5-small...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Model loaded! Parameters: 75,519,360


In [4]:
@torch.inference_mode()
def diacritize(text, max_length=512):
    # FLAN-T5 might benefit from instruction prefix
    prompt = f"Add Arabic diacritics: {text}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=max_length, num_beams=4, early_stopping=True)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    # If model returns instruction, extract Arabic text
    if ':' in result and not any(c in result for c in '\u0621\u0622\u0623'):
        return text  # Fallback
    return result

In [5]:
# Metrics
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}

def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        if ARABIC_LETTER_PATTERN.match(text[i]):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((text[i], ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth, exclude_irab=False):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, ser_sum, n = 0, 0, 0, 0, 0, 0
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_text, ref_text = pred['text_diacritized'], gt_lookup[pred['id']]
        pred_pairs, ref_pairs = extract_diacritics(pred_text), extract_diacritics(ref_text)
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            pred_d, ref_d = pred_pairs[i][1], ref_pairs[i][1]
            if exclude_irab:
                pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
                ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
            if pred_d != ref_d: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        pred_words, ref_words = pred_text.split(), ref_text.split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        if pred_text != ref_text: ser_sum += 1
        n += 1
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0, 'SER': ser_sum/n if n else 0, 'n_samples': n}

## Dev Set Inference

In [6]:
with open(DEV_INPUT, 'r', encoding='utf-8') as f: dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f: dev_output = json.load(f)
print(f"Dev samples: {len(dev_input)}")

dev_predictions = []
for item in tqdm(dev_input, desc="Dev Set"):
    try:
        diacritized = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    except Exception as e:
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})

Dev samples: 260


Dev Set: 100%|██████████| 260/260 [02:05<00:00,  2.06it/s]


In [7]:
print("="*60 + "\nDEV SET RESULTS\n" + "="*60)
m1 = compute_metrics(dev_predictions, dev_output, False)
print(f"\n[With I'rab] DER: {m1['DER']*100:.2f}% | WER: {m1['WER']*100:.2f}% (PRIMARY) | SER: {m1['SER']*100:.2f}%")
m2 = compute_metrics(dev_predictions, dev_output, True)
print(f"[No I'rab]   DER: {m2['DER']*100:.2f}% | WER: {m2['WER']*100:.2f}% | SER: {m2['SER']*100:.2f}%")

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

DEV SET RESULTS

[With I'rab] DER: 28.97% | WER: 98.94% (PRIMARY) | SER: 99.62%
[No I'rab]   DER: 28.63% | WER: 98.94% | SER: 99.62%


## Blind Test Inference

In [9]:
with open(TEST_INPUT, 'r', encoding='utf-8') as f: test_input = json.load(f)
print(f"Test samples: {len(test_input)}")

test_predictions = []
for item in tqdm(test_input, desc="Test Set"):
    try:
        diacritized = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    except Exception as e:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"\nSUBMISSION READY: {zip_file} ({zip_file.stat().st_size/1024:.1f} KB)")

Test samples: 328


Test Set: 100%|██████████| 328/328 [08:36<00:00,  1.57s/it]


SUBMISSION READY: ./submissions/flan_t5_submission_20260220_103619.zip (18.9 KB)
